In [61]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [62]:
# Constants
TRAIN_SIZE = 0.2
RANDOM_STATE = 1082

In [64]:
# Load the dataset
base_dir = "../../../Datasets/csv/UAVATTACK/"
real_data_path = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
synthetic_files = [
    "spoofing-CTGAN-synthetic.csv",
    "spoofing-GaussianCopula-synthetic.csv",
    "spoofing-TVAE-synthetic.csv",
    "syntheticSpoofing_cgan_data.csv",
    "syntheticSpoofing_gan_data.csv"
]
# Load synthetic datasets
synthetic_dfs = {}
for file_name in synthetic_files:
    file_path = os.path.join(base_dir, file_name)
    try:
        synthetic_dfs[file_name] = pd.read_csv(file_path)
        print(f"Loaded {file_name}")
    except Exception as e:
        print(f"Could not load {file_name}: {e}")

# Load real data
try:
    real_df = pd.read_csv(real_data_path)
    real_df.columns = real_df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None

In [66]:
# Preprocess real data
if real_df is not None:
    if 'label' not in real_df.columns:
        raise ValueError("Real data does not have 'label' column.")
    X_real = real_df.drop(columns=[col for col in ['label', 'timestamp'] if col in real_df.columns])
    y_real = real_df['label']
    if y_real.dtype == object or y_real.dtype.name == 'category':
        le_real = LabelEncoder()
        y_real = le_real.fit_transform(y_real)
else:
    raise RuntimeError("Real data could not be loaded.")

In [67]:
# Models to evaluate
models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=5000, solver='saga'),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(100, 50, 25), activation='tanh', solver='adam',
                                    random_state=RANDOM_STATE, learning_rate='adaptive', early_stopping=True, ),
    "XGBoost": XGBClassifier(objective='binary:logistic', random_state=RANDOM_STATE, eval_metric='mae', )
}

In [68]:
for synth_name, synth_df in synthetic_dfs.items():
    print(f"Training on synthetic: {synth_name}, testing on real data")

    if 'label' not in synth_df.columns:
        print(f"Skipping {synth_name}: 'label' column not found.")
        continue

    # Drop label and timestamp from synthetic, match columns to real data
    X_synth = synth_df.drop(columns=[col for col in ['label', 'timestamp'] if col in synth_df.columns])
    y_synth = synth_df['label']
    if y_synth.dtype == object or y_synth.dtype.name == 'category':
        le_synth = LabelEncoder()
        y_synth = le_synth.fit_transform(y_synth)

    X_synth_train, _, y_synth_train, _ = train_test_split(
        X_synth, y_synth, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_synth
    )

    # Scale based on synthetic training split, apply to both train and real test
    scaler = StandardScaler()
    X_synth_train_scaled = scaler.fit_transform(X_synth_train)
    X_real_scaled = scaler.transform(X_real)

    for model_name, model in models.items():
        print(f"\n{model_name}")
        print("Training on synthetic")
        model.fit(X_synth_train_scaled, y_synth_train)
        print("Evaluating on real data...")
        y_pred = model.predict(X_real_scaled)
        acc = accuracy_score(y_real, y_pred)
        print(f"Accuracy: {acc:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_real, y_pred, zero_division=0))
        print("Confusion Matrix:")
        print(confusion_matrix(y_real, y_pred))

Training on synthetic: spoofing-CTGAN-synthetic.csv, testing on real data

Logistic Regression
Training on synthetic
Evaluating on real data...
Accuracy: 0.9743

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      3124
           1       1.00      0.81      0.90       498

    accuracy                           0.97      3622
   macro avg       0.99      0.91      0.94      3622
weighted avg       0.98      0.97      0.97      3622

Confusion Matrix:
[[3124    0]
 [  93  405]]

MLP Classifier
Training on synthetic
Evaluating on real data...
Accuracy: 0.9845

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3124
           1       1.00      0.89      0.94       498

    accuracy                           0.98      3622
   macro avg       0.99      0.94      0.97      3622
weighted avg       0.98      0.98      0.98      3622

Confusion 